In [1]:
# --- [CELL 0]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 1}
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt # plotting
import numpy as np # linear algebra
import os # accessing directory structure
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import clip
from PIL import Image
from collections import Counter
from sklearn import preprocessing
from sklearn.preprocessing import OneHotEncoder

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
seed=42

In [2]:
# --- [CELL 1]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 2}
#use this block later to read cvs hopefully :)
train_df=pd.read_csv("data_small/train.csv",index_col=0)
#
train_labels=train_df['label'].to_numpy()
train_labels=train_labels.reshape(train_labels.shape[0],1)

#vocab=train_df['label'].to_numpy()
train_df=train_df.drop(columns=['label', 'label_type']) #train_df=train_df.drop(columns=('label')) # for reproducing and fixing
test_df=pd.read_csv("data_small/test.csv",index_col=0)
#
test_labels=test_df['label'].to_numpy()
test_labels=test_labels.reshape(test_labels.shape[0],1)

test_df=test_df.drop(columns=('label'))
val_df = pd.read_csv("data_small/val.csv",index_col=0)
#
val_labels=val_df['label'].to_numpy()
val_labels=val_labels.reshape(val_labels.shape[0],1)

val_df = val_df.drop(columns=('label'))
vocab=np.append(train_labels,val_labels)
#print(vocab.shape)
vocab=np.unique(vocab)
vocab=vocab.reshape(vocab.shape[0],1)
print(vocab.shape)
oh = OneHotEncoder(sparse_output=False)
hot_vocab=oh.fit_transform(vocab)
train_df.shape,val_df.shape,test_df.shape
train_df

(393, 1)


,0,1,2,3,4,5,6,7,8,9,...,1526,1527,1528,1529,1530,1531,1532,1533,1534,1535
0,0.080583,0.792151,0.080401,-1.041164,-0.493019,0.754633,0.179797,-0.569844,0.252744,0.158662,...,0.196967,-0.737442,0.163604,0.062471,-0.217384,0.596988,0.223638,0.196604,0.887675,0.464498
1,0.455594,1.274032,-0.631216,0.794531,0.099354,0.822643,0.812349,0.094368,-0.177235,-0.248552,...,-0.389059,-0.171184,-0.183861,-0.289215,0.174507,0.070595,-0.419749,0.100051,0.518568,1.237458
2,-0.552260,-0.038321,-0.325835,0.782001,-0.532989,1.414179,0.223727,-0.842841,0.667259,-0.089491,...,0.032783,-0.282632,0.193739,-0.262225,0.482379,-0.083215,0.127543,-0.138698,0.542505,0.421897
3,0.268696,0.775190,0.135756,0.377669,-0.244280,0.042453,0.659636,-0.074960,-0.386696,-1.113595,...,-0.194296,-0.149648,-0.560157,-0.220542,0.892043,0.913104,0.060146,0.185875,0.235500,0.044229
4,0.503005,0.255617,0.234536,-0.677124,-0.528914,0.217532,-0.165425,-0.200084,-0.689785,-0.161943,...,-0.136319,-0.062786,-0.010574,-0.419187,0.453835,0.638273,0.142632,-0.036917,0.582661,0.286633
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
715,1.033454,0.902746,-0.119601,0.323484,-0.056853,0.130698,0.138174,0.283947,-0.036717,-0.617271,...,-0.580590,-0.459752,0.163029,-0.434243,-0.344340,0.498533,-0.678874,-0.103662,0.209339,-0.305093
716,0.832794,0.483795,0.523217,0.253099,-0.225674,0.378319,-0.162106,-0.040049,0.013995,-0.131889,...,-0.004563,-0.234538,-0.198814,-0.853732,0.701991,0.815769,-0.068685,-0.259593,0.543328,0.576517
717,-0.553783,-0.024596,0.087584,0.098981,0.350917,0.491366,0.607917,-0.157551,-0.179839,-0.848238,...,-0.168125,-0.284248,-0.149445,-0.542359,0.729477,0.843604,-0.120873,-0.157058,0.500746,0.462701
718,0.349448,0.790657,0.125537,-0.923095,-0.001576,0.381703,-0.252967,-0.072247,-0.181222,0.535579,...,-0.718043,-0.518717,0.226134,-0.599162,0.097526,0.494871,-0.301887,0.009932,0.839774,0.340723


In [3]:
# --- [CELL 2]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 3}
train_arr=train_df.to_numpy()
train_arr=torch.from_numpy(train_arr)
test_arr=test_df.to_numpy()
test_arr=torch.from_numpy(test_arr)
val_arr=val_df.to_numpy()
val_arr=torch.from_numpy(val_arr)


train_labels=oh.transform(train_labels)
val_labels =oh.transform(val_labels)
test_enc_labels=[]
for i in range(test_labels.shape[0]):
    try:
        test_enc_labels.append(oh.transform(test_labels[i]))
    except ValueError as e:
        z=np.zeros((1,6294))
        test_enc_labels.append(z)
test_labels=np.array(test_enc_labels)
test_labels=np.squeeze(test_labels)
test_labels.shape

train_labels=torch.tensor(train_labels)
train_labels=train_labels.to(torch.float32)
val_labels=torch.tensor(val_labels)
val_labels=val_labels.to(torch.float32)
test_labels=torch.tensor(test_labels)
test_labels=test_labels.to(torch.float32)

In [4]:
# --- [CELL 3]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 4}
class myDataset(Dataset):
    def __init__(self, array,labels):
        self.array = array.to(device)
        self.label = labels.to(device)
          # stuff
      
    def __getitem__(self, index):
        # stuff
        data=self.array[index]
        data=data.to(torch.float32)
        label=self.label[index].type(torch.float32)
        label=self.label[index].to(device)
        #print("hello this is the whol out put tensore i should be 19000")
        #print(self.label.shape)
        return data, label

    def __len__(self):
        return len(self.array) # of how many examples(images?) you have

In [5]:
# --- [CELL 4]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 5}
device = "cuda" if torch.cuda.is_available() else "cpu"

In [6]:
# --- [CELL 5]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 6}
customDataset=myDataset(train_arr,train_labels)
train_dataloader = DataLoader(customDataset, batch_size=64,shuffle=True, num_workers=0)

In [7]:
# --- [CELL 6]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 7}
#trial torch model
class AnswerModel(torch.nn.Module):

    def __init__(self):
        super(AnswerModel, self).__init__()
        
        self.norm0 = torch.nn.LayerNorm(1536).to(device)
        self.dropout0 = torch.nn.Dropout(0.5).to(device)
        self.linear1 = torch.nn.Linear(1536, 512).to(device)
        #check layer norm
        self.norm1 = torch.nn.LayerNorm(512).to(device)
        self.dropout1 = torch.nn.Dropout(0.5).to(device)
        
        self.activation = torch.nn.ReLU().to(device)
        
        self.linear2 = torch.nn.Linear(512 , 6294).to(device)
        
        self.aux = torch.nn.Linear(512,4).to(device)
        self.dropout1 = torch.nn.Dropout(0.5).to(device)
        self.gate = torch.nn.Linear(4, 6294).to(device)
        self.sigmoid=torch.nn.Sigmoid().to(device)
        
        
    def forward(self, x):
        x = self.norm0(x).to(device)
        x = self.dropout0(x).to(device)
        
        x= self.linear1(x).to(device)
        x = self.dropout1(x).to(device)
        
        xaux =self.aux(x).to(device)
        xaux =self.gate(xaux).to(device)
        vqa = self.linear2(x).to(device)
        out = vqa * self.sigmoid(xaux)
        return out,xaux
model= AnswerModel().to(device)
print(model)

AnswerModel(
  (norm0): LayerNorm((1536,), eps=1e-05, elementwise_affine=True)
  (dropout0): Dropout(p=0.5, inplace=False)
  (linear1): Linear(in_features=1536, out_features=512, bias=True)
  (norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True)
  (dropout1): Dropout(p=0.5, inplace=False)
  (activation): ReLU()
  (linear2): Linear(in_features=512, out_features=6294, bias=True)
  (aux): Linear(in_features=512, out_features=4, bias=True)
  (gate): Linear(in_features=4, out_features=6294, bias=True)
  (sigmoid): Sigmoid()
)


In [8]:
# --- [CELL 7]: ---
# cell_state: edited
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 8}
# === BEFORE (original) ===
# #1 epoch
# def run_model(model,dataloader, optimizer,train = True ):
#     if train:
#         model.train()
#   
#     pred = []
#     True_labels = []
#     loss = torch.nn.CrossEntropyLoss()
#     #loss_aux = torch.nn.CrossEntropyLoss()
#     total_loss = 0
#     for (data, label) in dataloader: 
#         
#         data=data.to(device)
#         label=label.to(device)
#         #print("!!!!!!!!!!!!!!PLS!!!!!!!!!!!!!!!!")
#         #print(next(model.parameters()).is_cuda)
#         #print("!!!!!!!!!!!!!!!DATALOCATION!!!!!!!!!!!!!!!!!")
#         #print(data.device)
#         optimizer.zero_grad()
#         output,out_aux = model(data)
#         output=output.type(torch.FloatTensor).to(device)
#         out_aux=out_aux.type(torch.FloatTensor).to(device)
#         #print("output shape is,",output.shape)
#         #print("label shape is,",label.shape)
#    
#         loss_ = loss(output, label).to(device)
#         loss_aux=loss(out_aux,label).to(device)
#         mod_loss = loss_+loss_aux 
#         mod_loss.backward()
#         total_loss+=mod_loss.item()
#         
#         optimizer.step()
#         pred.append(output)
#         True_labels.append(label)
#         #print("total loss",total_loss)
#         
#     return pred ,True_labels, total_loss/len(dataloader)

# === AFTER (edited) ===
def run_model(model,dataloader, optimizer,train = True ):
    if train:
        model.train()

    pred = []
    True_labels = []
    ce_main = torch.nn.CrossEntropyLoss()
    ce_aux = torch.nn.CrossEntropyLoss()

    total_loss = 0
    for (data, label) in dataloader:

        data=data.to(device)
        label=label.to(device)

        # Convert one-hot labels to class indices for CrossEntropyLoss
        label_idx = torch.argmax(label, dim=1).long()

        optimizer.zero_grad()
        output,out_aux = model(data)
        output=output.to(device)
        out_aux=out_aux.to(device)

        loss_ = ce_main(output, label_idx)
        # Aux head has 4 logits; map labels into 4 groups to match shape
        aux_target = (label_idx % 4).long()
        loss_aux = ce_aux(out_aux, aux_target)
        mod_loss = loss_+loss_aux
        mod_loss.backward()
        total_loss+=mod_loss.item()

        optimizer.step()
        pred.append(output.detach())
        True_labels.append(label.detach())


    return pred ,True_labels, total_loss/len(dataloader)

In [9]:
# --- [CELL 8]: ---
# cell_state: unchanged
# execution_status: {'status': 'ok', 'done': True, 'execution_count': 9}
epoch = 2 #150

optimizer = torch.optim.Adam(model.parameters(), 0.001, weight_decay=.01)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=.1, threshold=1e-6)

for e in range(epoch):
    pred,labels,loss = run_model(model,train_dataloader,optimizer)
    #training accuracy
    correct=0
    for i in range(len(pred)):
        predictions = pred[i].to(device)
        t_label = labels[i].to(device)
        position = torch.argmax(predictions).to(device)
        pos_label= torch.argmax(t_label).to(device)
        #print("pred",position)
        #print("true",pos_label)
        if (position == pos_label ):
            correct+=1
    scheduler.step(loss)
    print("epoch : ",e)
    print("training accuracy is ",correct/len(pred)*1.0)
  # calculate acc, f1 score, recall ......
    print(loss)

epoch :  0
training accuracy is  0.0
15.208370844523111
epoch :  1
training accuracy is  0.08333333333333333
13.456032355626425


In [10]:
import numpy as np
import torch

assert 'pred' in globals() and len(pred) > 0, 'Training loop did not produce predictions.'
assert 'labels' in globals() and len(labels) == len(pred), 'Predictions/labels batch lists must align.'
assert 'loss' in globals(), 'Training loop did not produce a loss value.'
assert np.isfinite(loss), 'Loss is non-finite.'

for i, (p, y) in enumerate(zip(pred, labels)):
    assert isinstance(p, torch.Tensor) and isinstance(y, torch.Tensor), f'Batch {i}: pred/label must be tensors.'
    assert p.ndim == 2, f'Batch {i}: predictions must be 2D logits [N, C].'
    assert torch.isfinite(p).all().item(), f'Batch {i}: logits contain NaN/Inf.'
    assert y.ndim == 2, f'Batch {i}: unexpected label rank {y.ndim}'
    assert y.shape[0] == p.shape[0], f'Batch {i}: batch size mismatch.'
    assert torch.isfinite(y).all().item(), f'Batch {i}: labels contain NaN/Inf.'
    assert torch.allclose(y.sum(dim=1), torch.ones(y.shape[0], device=y.device)), \
        f'Batch {i}: one-hot labels must sum to 1.'